# 사전준비

In [ ]:
# 1. cuda-toolkit 12.6 설치 완료(nvcc --version)
# 2. pyproject.toml에 있는 transformers, datasets 지우기 (uv remove transformers datasets)
# 3. pyroject.toml 파일 업데이트 
# ```
# [tool.uv.sources]
# torch = { index = "pytorch" }
# torchvision = { index = "pytorch" }
# unsloth = { git = "https://github.com/unslothai/unsloth.git" }
# unsloth-zoo = { git = "https://github.com/unslothai/unsloth-zoo.git" }

# [[tool.uv.index]]
# name = "pytorch"
# url = "https://download.pytorch.org/whl/cu126"
# explicit = true
# ```
# 4. unsloth 설치 (uv add unsloth unsloth-zoo)
# 5. triton 설치 (uv pip install https://github.com/woct0rdho/triton-windows/releases/download/v3.2.0-windows.post10/triton-3.2.0-cp312-cp312-win_amd64.whl --no-deps)

# 1. 모델 불러오기

## 1) 베이스 모델 불러오기

In [ ]:
from unsloth import FastLanguageModel 
import torch 

max_seq_length = 1024    # ✅ 한 번에 처리할 최대 토큰 수 (데이터 길이에 맞게 조정, 클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장, 변경 불필요)
load_in_4bit = True      # ✅ 4bit 압축 로드 여부 (True = VRAM 절약, VRAM 부족 시 반드시 True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",  # ✅ 사용할 베이스 모델 이름 (허깅페이스 모델 ID)
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")
print(f"   dtype : {next(model.parameters()).dtype}")
print(f"   device: {next(model.parameters()).device}")

## 2) LoRA 어댑터 준비

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # ✅ LoRA rank (클수록 학습 용량 많음, 보통 8·16·32·64 중 선택)
    target_modules = [           # 어댑터 붙일 레이어 선택
        # 💡 Instruct 파인튜닝은 경험적으로 "q_proj", "v_proj" 두 개만으로도 충분한 효율을 냄
        #    그러나 실습에서는 전체 레이어를 포함해 학습 효과를 눈에 띄게 확인하기 위해 모두 포함
        #    VRAM이 빡빡하거나 빠른 실험이 필요하다면 q_proj, v_proj만 남겨도 무방
        "q_proj", "k_proj",      #   └─ 어텐션 레이어
        "v_proj", "o_proj",
        "gate_proj", "up_proj",  #   └─ MLP 레이어
        "down_proj",
    ],
    lora_alpha = 16,             # ✅ LoRA 학습 강도 (보통 r과 동일하게 설정)
    lora_dropout = 0,            # 뉴런 드롭 비율 (0 = Unsloth 최적화, 변경 불필요)
    bias = "none",               # 바이어스 학습 여부 ("none" = Unsloth 최적화, 변경 불필요)
    use_gradient_checkpointing = "unsloth",  # VRAM 절약 방식 ("unsloth" 권장, 변경 불필요)
    random_state = 3407,         # 재현성을 위한 랜덤 시드 (결과 재현 시 동일 값 유지)
    use_rslora = False,          # Rank Stabilized LoRA 사용 여부 (대용량 모델 학습 시 True 고려)
    loftq_config = None,         # LoftQ 양자화 설정 (실습에서는 불필요)
)

print("✅ LoRA 어댑터 추가 완료!")

# 2. 데이터셋 불러오기

## 1) 데이터셋 로드

In [18]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="data/custom_dataset.jsonl",
)

print(f"데이터 수: {len(dataset)}")
print(f"컬럼: {dataset.column_names}")
print(dataset)

데이터 수: 1
컬럼: {'train': ['messages']}
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 56
    })
})


## 2) 데이터 분할

In [26]:
split = dataset["train"].train_test_split(test_size=0.2, seed=42)
split

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 44
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 12
    })
})

In [27]:
train_dataset = split["train"]
valid_dataset = split["test"]

In [28]:
train_dataset[0]

{'messages': [{'role': 'system',
   'content': '당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.'},
  {'role': 'user', 'content': '주문 후 결제 수단을 바꿀 수 있나요?'},
  {'role': 'assistant',
   'content': '결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!'}]}

In [22]:
tokenizer.apply_chat_template(
    [
        {'role': 'system','content': '당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.'},
        {'role': 'user', 'content': '주문 후 결제 수단을 바꿀 수 있나요?'},
        {'role': 'assistant','content': '결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!'}
    ],
    tokenize=False,
    add_generation_prompt=False
)

'<|im_start|>system\n당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.<|im_end|>\n<|im_start|>user\n주문 후 결제 수단을 바꿀 수 있나요?<|im_end|>\n<|im_start|>assistant\n결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!<|im_end|>\n'

## 2) formatting_func 만들기

In [23]:
def formatting_prompts_func(data):
    result = []
    messages = data["messages"]  # 이미 하나의 샘플

    # messages가 딕셔너리 리스트인지 확인
    if isinstance(messages[0], dict):
        # 단일 샘플로 들어온 경우
        chat = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        result.append(chat)
    else:
        # 배치로 들어온 경우
        for msgs in messages:
            chat = tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=False
            )
            result.append(chat)
    return result

# 3. Trainer 만들기

## 1) TrainingArgument

In [ ]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

args = TrainingArguments(
    per_device_train_batch_size = 2,         # ✅ GPU당 배치 크기 (VRAM에 맞게 조정, OOM 시 줄이기)
    gradient_accumulation_steps = 4,          # ✅ 실질 배치 = 2×4 = 8 (배치 크기 줄인 만큼 스텝 수 늘리기)
    warmup_steps = 5,                         # lr 워밍업 스텝 수 (전체 스텝의 5~10% 권장)
    # max_steps = 100,                        # ✅ 빠른 테스트용 (전체 학습 시 num_train_epochs 사용)
    num_train_epochs = 10,                    # ✅ 전체 에포크 수 (EarlyStopping 사용 시 넉넉하게 설정)
    learning_rate = 2e-4,                     # ✅ 학습률 (LoRA 권장값: 1e-4 ~ 3e-4)
    fp16 = not is_bfloat16_supported(),       # 구형 GPU용 (자동 감지, 변경 불필요)
    bf16 = is_bfloat16_supported(),           # 최신 GPU용 (자동 감지, 변경 불필요)
    logging_steps = 1,                        # 몇 스텝마다 손실(loss)을 출력할지
    optim = "adamw_8bit",                     # VRAM 절약형 옵티마이저 (변경 불필요)
    weight_decay = 0.01,                      # 과적합 방지 정규화 (보통 0.01 고정)
    lr_scheduler_type = "linear",             # lr 감소 방식 (linear·cosine 중 선택 가능)
    seed = 1234,                              # 재현성을 위한 랜덤 시드
    output_dir = "outputs",                   # ✅ 체크포인트 저장 폴더 (경로 직접 지정)
    report_to = "none",                       # 로그 기록 위치 (wandb 연동 시 "wandb"로 변경)
    average_tokens_across_devices = False,    # 멀티 GPU 토큰 평균화 (단일 GPU 시 False)
    load_best_model_at_end = True,            # EarlyStopping 사용 시 필수 (없으면 에러 발생)
    eval_strategy="epoch",                    # ✅ 검증 주기 (epoch마다 val loss 계산)
    save_strategy="epoch"                     # ✅ 체크포인트 저장 주기 (eval_strategy와 맞춰야 함)
)

## 2) Early Stopping

In [30]:
from transformers import EarlyStoppingCallback

callbacks = [
    EarlyStoppingCallback(
        early_stopping_patience = 3   # 3번 연속 안 좋아지면 멈춤
    )
]

## 3) SFTTrainer

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,              # ✅ 학습에 사용할 데이터셋 (직접 준비한 데이터 지정)
    eval_dataset = valid_dataset,               # ✅ 검증에 사용할 데이터셋 (EarlyStopping 기준)
    formatting_func = formatting_prompts_func,  # ✅ 데이터를 모델 입력 형식으로 변환하는 함수
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,                       # 전처리 CPU 코어 수 (CPU 코어에 맞게 조정)
    packing = False,                            # ✅ 멀티턴 데이터 학습 시 False 권장
    args = args,
    callbacks=callbacks                         # EarlyStopping 콜백 (학습 조기 종료)
)

print("✅ 트레이너 설정 완료!")

# 4. 학습

## 1) 학습 전 GPU 현황

In [32]:
# 학습 전 GPU 메모리 상태 기록 (학습 후 셀과 비교용)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU 이름     : {gpu_stats.name}")
print(f"전체 VRAM    : {max_memory} GB")
print(f"현재 예약량  : {start_gpu_memory} GB")
print(f"남은 여유    : {round(max_memory - start_gpu_memory, 3)} GB")

GPU 이름     : NVIDIA GeForce RTX 4070 Laptop GPU
전체 VRAM    : 7.996 GB
현재 예약량  : 3.02 GB
남은 여유    : 4.976 GB


## 2) 학습하기

In [33]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44 | Num Epochs = 10 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Epoch,Training Loss,Validation Loss
1,2.058600,1.946658
2,1.358000,1.286548
3,1.071700,1.087540
4,0.775200,1.030417
5,0.804700,1.018508
6,0.711000,1.017988
7,0.517100,1.033422
8,0.407500,1.058038
9,0.447300,1.073278


In [34]:
# 학습 후 GPU 메모리 및 시간 통계
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"✅ 학습 완료!")
print(f"")
print(f"⏱  학습 시간        : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} 분")
print(f"")
print(f"💾 전체 VRAM 사용량 : {used_memory} GB ({used_percentage} %)")
print(f"💾 LoRA 학습 사용량 : {used_memory_for_lora} GB ({lora_percentage} %)")
print(f"   (모델 로드 제외, 순수 학습에 쓴 VRAM)")

✅ 학습 완료!

⏱  학습 시간        : 2.86 분

💾 전체 VRAM 사용량 : 3.467 GB (43.359 %)
💾 LoRA 학습 사용량 : 0.447 GB (5.59 %)
   (모델 로드 제외, 순수 학습에 쓴 VRAM)


## 3) 모델 저장

In [ ]:
# 학습 후 16bit로 병합 저장
model.save_pretrained_merged(
    "C:/potenup3/models/model_merged_qwen_custom",  # ✅ 저장 폴더 경로 (한글 경로 사용 불가)
    tokenizer,
    save_method="merged_16bit"   # ✅ 저장 방식 (merged_16bit: 추론용 완전 병합 / lora: 어댑터만 저장)
)

# 5. 추론

## 1) 모델 불러온 후 추론

In [1]:
# unsloth Bug : 현재 FastLanguageModel로 추론하는 것은 버그 존재
# 이때 model에 unsloth가 반영되어 있기 때문에 재시작 한 후에 불러와야 한다.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ── 병합된 모델 로드 ──────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    "C:/potenup3/models/model_merged_qwen_custom",
    dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("C:/potenup3/models/model_merged_qwen_custom")

print("✅ 모델 로드 완료!")

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0330 17:58:43.276000 39576 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
The tokenizer you are loading from 'C:/potenup3/models/model_merged_qwen_custom' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ 모델 로드 완료!


In [2]:
input_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "테디노트 유튜브 채널에 대해 알려주세요"}],
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
print(inputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198, 130229,  89235, 127121,
          28626, 126310, 144293, 131196,   3315,    109,    226, 139287,  19391,
         131978, 138630,  91669, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [ ]:
# AttributeError: 'Qwen2Attention' object has no attribute 'apply_qkv' -> 커널 재시작 후 모델 다시 불러와주세요
# model 속에 unsloth가 반영되어있어 이를 없애기 위해 재시작이 필요합니다.
from transformers import TextStreamer

print("=== 학습 후 출력 (After) ===")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,          # ✅ 생성할 최대 토큰 수 (응답 길이 조절)
    use_cache=False,             # 캐시 사용 여부 (스트리밍 시 False 권장, 변경 불필요)
    repetition_penalty=1.3,      # ✅ 같은 표현 반복 방지 (1.0=없음, 높을수록 다양한 표현)
    temperature=0.7,             # ✅ 출력 다양성 조절 (낮을수록 일관성↑, 높을수록 창의성↑)
    do_sample=True,              # ✅ 샘플링 방식 (True=확률 기반 다양한 출력, False=greedy)
)

# 2) Ollama에 내 모델 등록하기

## 1) Unsloth로 모델 불러오기

In [4]:
from unsloth import FastLanguageModel 
import torch 

max_seq_length = 1024    # 한 번에 처리할 최대 토큰 수 (클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장)
load_in_4bit = False      # 4bit 압축 로드 여부 (True = VRAM 절약) ✅

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "C:/potenup3/models/model_merged_qwen_custom",
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


C:\Users\user\AppData\Local\Temp\ipykernel_39576\1772083609.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.


✅ 모델 로드 완료!


## 2) GGUF로 모델 저장하기

In [ ]:
# GGUF 변환 + 저장
# cmake installer, visual studio installer, openssl가 자동으로 뜸
# https://cmake.org/download/
# https://visualstudio.microsoft.com/ko/downloads/
# https://openssl-library.org/source/
# vscode 껏다 켜기
model.save_pretrained_gguf(
    "C:/potenup3/models/model_merged_qwen_custom_gguf",           # ✅ 저장 폴더 경로 (한글 경로 사용 불가)
    tokenizer,
    quantization_method = "q4_k_m"        # ✅ 양자화 방식 (q4_k_m: 속도·품질 균형, q8_0: 고품질·큰 용량)
)

In [ ]:
# GGUF 변환 + 저장
# cmake installer, visual studio installer, openssl가 자동으로 뜸
# https://cmake.org/download/
# https://visualstudio.microsoft.com/ko/downloads/
# https://openssl-library.org/source/
# vscode 껏다 켜기
model.save_pretrained_gguf(
    "C:/potenup3/models/model_merged_qwen_custom_gguf_q8",           # ✅ 저장 폴더 경로 (한글 경로 사용 불가)
    tokenizer,
    quantization_method = "q8_0"        # ✅ 양자화 방식 (q8_0: 고품질·큰 용량, q4_k_m: 속도·품질 균형)
)

## 3) Ollama에 내 모델 GGUF 등록하기

In [ ]:
# 터미널에서 실행하세요 
# cd models/model_merged_gguf
# ollama create 내모델이름설정 -f Modelfile 
# ollama Desktop 켜보세요